# GamePulse — 01 · Bronze Layer
## Raw Mock Data Generation

**SYST52461 – Big Data Storage and Analysis**  
**Term Project · Winter 2026**

---

### Purpose
This notebook generates realistic mock data for the **GamePulse** simulated online gaming platform  
and writes it to four Bronze Delta tables in `sandbox_catalog.gamepulse`.

### Tables Created
| Table | Description | Rows |
|---|---|---|
| `players_bronze` | Player demographics and account info | 500 |
| `games_bronze` | Game catalogue with genre and mode | 30 |
| `game_sessions_bronze` | Session logs per player per game | 3 000 |
| `purchases_bronze` | In-game transactions per player | 2 000 |

### Intentional Data Quality Issues (for Silver layer to fix)
- Mixed-case and abbreviated `region` values (`"NA"`, `"north america"`, `"North America"`)
- Mixed-case `account_type` values (`"premium"`, `"PREMIUM"`, `"Premium"`)
- `age` stored as a **string** instead of an integer
- ~6% null values in `email`
- ~5% null values in `score` (session ended abnormally)
- Negative `amount_spent` values (~3%) — data entry errors
- `session_date` and `purchase_date` stored as strings
- `duration_minutes` occasionally 0 or negative (~2%)

## 1. Environment Setup

In [ ]:
import logging
import random
from datetime import datetime, timedelta

from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

# ── Logging ────────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("gamepulse.bronze")

# ── Catalog / Schema constants ─────────────────────────────────────────────────
CATALOG = "sandbox_catalog"
SCHEMA  = "gamepulse"

# ── Reproducibility ────────────────────────────────────────────────────────────
random.seed(42)

log.info("Imports complete. Catalog=%s  Schema=%s", CATALOG, SCHEMA)

## 2. Create Catalog and Schema

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA  IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE {CATALOG}.{SCHEMA}")

log.info("Schema ready: %s.%s", CATALOG, SCHEMA)

## 3. Helper Utilities

In [ ]:
def random_date(start: datetime, end: datetime) -> str:
    """Return a random date string (YYYY-MM-DD) between start and end."""
    delta = (end - start).days
    return (start + timedelta(days=random.randint(0, delta))).strftime("%Y-%m-%d")


def write_bronze(df, table_name: str) -> None:
    """Write a Spark DataFrame as a Delta table in the Bronze layer."""
    full_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    df.write.format("delta").mode("overwrite").saveAsTable(full_name)
    log.info("Bronze table written: %s  (%d rows)", full_name, df.count())


START = datetime(2023, 1, 1)
END   = datetime(2025, 12, 31)

log.info("Helper functions defined.")

---
## 4. Generate Mock Data

### 4.1 Table 1 — `players`

**Primary key:** `player_id`  
**Intentional issues:**
- `region` values inconsistently cased and abbreviated (`"NA"`, `"north america"`, `"North America"`)
- `account_type` inconsistently cased (`"premium"`, `"PREMIUM"`, `"Premium"`)
- `age` stored as string instead of integer
- ~6% null in `email`

In [ ]:
# ── Value pools with intentional inconsistencies ───────────────────────────────
# Different region formats to simulate inconsistent real-world data


REGIONS       = ["North America", "NA", "north america",
                 "Europe", "EU", "europe",
                 "Asia Pacific", "APAC", "asia pacific",
                 "Latin America", "LATAM", "latin america"]

# Mixed casing for account types to create standardization issues
ACCOUNT_TYPES = ["Free", "free", "FREE",
                 "Premium", "premium", "PREMIUM",
                 "Pro", "pro", "PRO"]

players_schema = StructType([
    StructField("player_id",    IntegerType(), False),
    StructField("username",     StringType(),  True),
    StructField("email",        StringType(),  True),   # ~6% null
    StructField("age",          StringType(),  True),   # stored as string — issue!
    StructField("region",       StringType(),  True),   # inconsistent casing / abbreviations
    StructField("account_type", StringType(),  True),   # inconsistent casing
    StructField("join_date",    StringType(),  True),   # stored as string — needs cast
])

players_data = []
for i in range(1, 501):
    email = f"player{i}@gamepulse.com" if random.random() > 0.06 else None   # ~6% null
    age   = str(random.randint(13, 55))                                        # stored as string
    players_data.append((
        i,
        f"user_{i:04d}",
        email,
        age,
        random.choice(REGIONS),
        random.choice(ACCOUNT_TYPES),
        random_date(START, END),
    ))

df_players_bronze = spark.createDataFrame(players_data, schema=players_schema)
print(f"players_bronze: {df_players_bronze.count()} rows")
display(df_players_bronze.limit(10))

### 4.2 Table 2 — `games`

**Primary key:** `game_id`  
**Intentional issues:**
- `genre` inconsistently cased (`"action"`, `"Action"`, `"ACTION"`)
- ~10% null in `game_mode`

In [ ]:
# Mixed casing genres to simulate inconsistent categorical data
GENRES     = ["Action", "action", "ACTION",
              "RPG", "rpg",
              "Strategy", "strategy", "STRATEGY",
              "Sports", "sports",
              "Puzzle", "puzzle",
              "Horror", "horror"]

# Include null values to simulate missing game mode data
GAME_MODES = ["Solo", "Multiplayer", "Co-op", None]   # ~10% null

GAME_TITLES = [
    "Shadow Realm", "Pixel Warriors", "Neon Drift", "Dungeon Lords", "Star Siege",
    "Frost Arena", "Turbo Clash", "Mystic Quest", "Iron Citadel", "Cyber Runners",
    "Blaze Protocol", "Ghost Squad", "Terra Wars", "Velocity Rush", "Dark Horizons",
    "Arena Kings", "Quantum Strike", "Storm Legends", "Nexus Point", "Solar Clash",
    "Blade Circuit", "Echo Protocol", "Titan Falls", "Void Runners", "Hyper Drive",
    "Dragon Siege", "Night Patrol", "Chaos Engine", "Rogue Path", "Warp Zone",
]

games_schema = StructType([
    StructField("game_id",   IntegerType(), False),
    StructField("title",     StringType(),  True),
    StructField("genre",     StringType(),  True),   # inconsistent casing
    StructField("game_mode", StringType(),  True),   # ~10% null
    StructField("release_year", IntegerType(), True),
])

games_data = [
    (
        i,
        GAME_TITLES[i - 1],
        random.choice(GENRES),
        random.choices(GAME_MODES, weights=[35, 40, 15, 10])[0],
        random.randint(2019, 2025),
    )
    for i in range(1, 31)
]

df_games_bronze = spark.createDataFrame(games_data, schema=games_schema)
print(f"games_bronze: {df_games_bronze.count()} rows")
display(df_games_bronze)

### 4.3 Table 3 — `game_sessions`

**Primary key:** `session_id`  
**Foreign keys:** `player_id` → `players`, `game_id` → `games`  
**Intentional issues:**
- ~5% null in `score`
- ~2% of `duration_minutes` is 0 or negative
- `session_date` stored as string

In [ ]:
OUTCOMES = ["Win", "Loss", "Draw", "Abandoned"]

sessions_schema = StructType([
    StructField("session_id",        IntegerType(), False),
    StructField("player_id",         IntegerType(), True),   # FK → players
    StructField("game_id",           IntegerType(), True),   # FK → games
    StructField("session_date",      StringType(),  True),   # stored as string
    StructField("duration_minutes",  DoubleType(),  True),   # ~2% invalid (<=0)
    StructField("score",             IntegerType(), True),   # ~5% null
    StructField("outcome",           StringType(),  True),
])

# ~2% invalid session durations (zero or negative)
# ~5% null scores to simulate incomplete session data
# Store session_date as string instead of date type
sessions_data = []
for i in range(1, 3001):
    # ~2% chance of invalid duration
    if random.random() < 0.02:
        duration = round(random.uniform(-5, 0), 1)
    else:
        duration = round(random.uniform(1, 180), 1)

    # ~5% null score
    score = random.randint(0, 10000) if random.random() > 0.05 else None

    sessions_data.append((
        i,
        random.randint(1, 500),    # player_id FK
        random.randint(1, 30),     # game_id FK
        random_date(START, END),
        duration,
        score,
        random.choice(OUTCOMES),
    ))

df_sessions_bronze = spark.createDataFrame(sessions_data, schema=sessions_schema)
print(f"game_sessions_bronze: {df_sessions_bronze.count()} rows")
display(df_sessions_bronze.limit(10))

### 4.4 Table 4 — `purchases`

**Primary key:** `purchase_id`  
**Foreign key:** `player_id` → `players`  
**Intentional issues:**
- ~3% of `amount_spent` is negative (data entry errors)
- `category` inconsistently cased
- `purchase_date` stored as string

In [ ]:
ITEM_CATEGORIES = ["Cosmetic", "cosmetic", "COSMETIC",
                   "Weapon", "weapon", "WEAPON",
                   "Booster", "booster",
                   "Character", "character",
                   "Currency", "currency"]

ITEM_NAMES = [
    "Dragon Skin", "Gold Pass", "XP Booster", "Epic Mount", "Legendary Sword",
    "Battle Armor", "Mystery Box", "Season Pass", "Avatar Frame", "Rare Emote",
    "Power Shield", "Victory Dance", "Storm Blade", "Crystal Pack", "Rainbow Trail",
    "Neon Wings", "Shadow Cloak", "Fire Wand", "Ice Crown", "Platinum Badge",
]

purchases_schema = StructType([
    StructField("purchase_id",   IntegerType(), False),
    StructField("player_id",     IntegerType(), True),   # FK → players
    StructField("item_name",     StringType(),  True),
    StructField("category",      StringType(),  True),   # inconsistent casing
    StructField("amount_spent",  DoubleType(),  True),   # ~3% negative
    StructField("purchase_date", StringType(),  True),   # stored as string
])

purchases_data = []
for i in range(1, 2001):
    # ~3% chance of negative amount (invalid record)
    if random.random() < 0.03:
        amount = round(-random.uniform(0.01, 20.0), 2)
    else:
        amount = round(random.uniform(0.99, 99.99), 2)

    purchases_data.append((
        i,
        random.randint(1, 500),         # player_id FK
        random.choice(ITEM_NAMES),
        random.choice(ITEM_CATEGORIES),
        amount,
        random_date(START, END),
    ))

df_purchases_bronze = spark.createDataFrame(purchases_data, schema=purchases_schema)
print(f"purchases_bronze: {df_purchases_bronze.count()} rows")
display(df_purchases_bronze.limit(10))

---
## 5. Write Bronze Tables to Delta

In [ ]:
write_bronze(df_players_bronze,  "players_bronze")
write_bronze(df_games_bronze,    "games_bronze")
write_bronze(df_sessions_bronze, "game_sessions_bronze")
write_bronze(df_purchases_bronze, "purchases_bronze")

log.info("All Bronze tables written successfully.")

---
## 6. Null Report — Verify Intentional Issues Were Preserved

In [ ]:
from pyspark.sql import functions as F

def null_report(df, label: str) -> None:
    """Print the count of null values for every column in a DataFrame."""
    null_counts = df.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns
    ])
    print(f"\n── Null Report: {label} ──")
    display(null_counts)

null_report(df_players_bronze,   "players_bronze")
null_report(df_games_bronze,     "games_bronze")
null_report(df_sessions_bronze,  "game_sessions_bronze")
null_report(df_purchases_bronze, "purchases_bronze")

---
## 7. Quick Preview of Each Bronze Table

In [ ]:
print("=== players_bronze ===")
spark.read.table(f"{CATALOG}.{SCHEMA}.players_bronze").printSchema()

print("=== games_bronze ===")
spark.read.table(f"{CATALOG}.{SCHEMA}.games_bronze").printSchema()

print("=== game_sessions_bronze ===")
spark.read.table(f"{CATALOG}.{SCHEMA}.game_sessions_bronze").printSchema()

print("=== purchases_bronze ===")
spark.read.table(f"{CATALOG}.{SCHEMA}.purchases_bronze").printSchema()

print("Bronze layer data generated with intentional quality issues for Silver layer processing.")